# 时间数据处理

pandas 的时间处理能力围绕 `Timestamp`（时间点）和 `DatetimeIndex`（时间索引）展开，核心流程：

1. **解析**：把字符串转成时间类型
2. **提取**：取出年/月/日/星期/季度等属性
3. **运算**：时间的加减、偏移
4. **重采样**：按频率升降采样
5. **时区**：时区转换（国内用得少，了解即可）


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
df = pd.read_csv(r'D:/dsml/data/weather.csv')
df.head(3)


## 字符串 -> 时间：pd.to_datetime()

`pd.to_datetime()` 是 pandas 时间处理的入口，把各种格式的日期字符串转成 `Timestamp` 或 `DatetimeIndex`。

```python
pd.to_datetime('2024-06-04')          # 单个字符串 -> Timestamp
pd.to_datetime(['2024-01', '2024-02']) # 列表 -> DatetimeIndex
pd.to_datetime(df['日期'])             # Series -> DatetimeIndex
```

**关键参数**：`format` 指定格式能大幅提速，`errors='coerce'` 把非法值转 NaN。

In [ ]:
# ===== pd.to_datetime() =====
df['日期'] = pd.to_datetime(df['日期'])
print('转换后类型:', df['日期'].dtype)
print(df[['日期', '城市', '温度(℃)']].head(5))

# 各种格式都能识别
formats = ['2024-06-04', '2024/06/04', '20240604', '2024-06-04 14:30:00', 'June 4, 2024']
print('\n各种日期格式 ->')
for s in formats:
    print(f'  {s:30s} -> {pd.to_datetime(s)}')

# errors='coerce'：非法日期变 NaT（Not a Time）
bad = pd.to_datetime(['2024-01-01', 'not_a_date', '2024-02-30'], errors='coerce')
print('\n容错处理 (errors=coerce):')
print(bad)


## 时间属性的提取

时间列为 `datetime64` 类型后，用 `.dt` 访问器提取各种属性。

Series 用 .dt，索引和单个时间点直接 .属性

| 属性 | 含义 |
|---|---|
| `.dt.year` | 年份 |
| `.dt.month` | 月份 (1-12) |
| `.dt.day` | 日 (1-31) |
| `.dt.hour` | 小时 |
| `.dt.dayofweek` | 星期几 (0=周一, 6=周日) |
| `.dt.weekday` | 同上 |
| `.dt.day_name()` | 星期几英文名 |
| `.dt.quarter` | 季度 (1-4) |
| `.dt.is_month_start` / `_end` | 是否月初/月末 |
| `.dt.days_in_month` | 当月天数 |
| `.dt.date` | 只取日期部分（去掉时分秒） |

In [ ]:
# ===== .dt 访问器提取时间属性 =====
df['年'] = df['日期'].dt.year
df['月'] = df['日期'].dt.month
df['日'] = df['日期'].dt.day
df['星期'] = df['日期'].dt.dayofweek
df['季度'] = df['日期'].dt.quarter
df['是否周末'] = df['星期'].isin([5, 6])
print(df[['日期', '年', '月', '日', '星期', '季度', '是否周末']].head(10))


## 时间索引：set_index + 日期列

把日期设为索引后，可以用日期字符串直接切片查询，极其方便。

```python
df.set_index('日期', inplace=True)
df.loc['2024-03']      # 整个 3 月
df.loc['2024-03-15']   # 3 月 15 日
df.loc['2024-01':'2024-03']  # 1~3 月
```

In [ ]:
# ===== 时间索引查询 =====
df_t = df.set_index('日期').sort_index()

print('--- 2024年3月 的数据 ---')
print(df_t.loc['2024-03'])

print('\n--- 2024-06-04 的数据 ---')
print(df_t.loc['2024-06-04'])

print('\n--- 4月1日~4月3日 ---')
print(df_t.loc['2024-04-01':'2024-04-03'])


## 重采样：resample()

把时间序列从一种频率变成另一种频率。`resample('规则')` 返回一个分组对象，接聚合函数。

### 频率规则（pandas 2.2+ 新版）

| 代码          | 含义 |
|-------------|---|
| `D`         | 每天 |
| `W`         | 每周（周日结束）|
| `ME`        | 月末（旧版 `M` 已废弃）|
| `MS`        | 月初 |
| `QE`        | 季末（旧版 `Q` 已废弃）|
| `YE`        | 年末（旧版 `Y`/`A` 已废弃）|
| `h`         | 每小时 |
| `T` / `min` | 每分钟 |
| `3D`        | 每 3 天 |
| `2W`        | 每 2 周 |

> 规律：带 E 后缀 = 周期结束（Month**E**nd, Quarter**E**nd, Year**E**nd）。
> 不带 = 周期开始（Month**S**tart, Quarter**S**tart, Year**S**tart）。

```python
df.resample('ME').mean()               # 按月均值
df.resample('W').agg({'温度': 'mean', '降水': 'sum'})  # 按周
```

In [ ]:
# ===== resample 重采样 =====
df_t = df.set_index('日期').sort_index()

monthly = df_t.resample('ME').agg({
    '温度(℃)': 'mean',
    '降水量(mm)': 'sum',
    '湿度(%)': 'mean'
})
#.agg():定义重取样后的属性如何计算
print('按月汇总:')
print(monthly.head(6))

weekly = df_t.resample('W').agg({
    '温度(℃)': ['mean', 'max', 'min'],
    '降水量(mm)': 'sum'
})
print('\n按周汇总:')
print(weekly.head(3))


## 位移：shift() / diff() / pct_change()

时间序列的经典操作：把数据在时间轴上挪位，用于计算同比、环比、差值。

| 方法 | 作用 |
|---|---|
| `shift(n)` | 平移 n 行（正数 = 往前翻（历史），负数 = 往后翻（未来））|
| `diff(n)` | n 阶差分 = 当前值 - n 行前值 |
| `pct_change(n)` | n 期变化率 = (当前-n行前)/n行前 |

```python
df['上期'] = df['值'].shift()        # 前一期的值
df['环比'] = df['值'].diff()          # 比上期多了多少
df['环比%'] = df['值'].pct_change()   # 环比增长率
```

In [ ]:
# ===== shift / diff / pct_change =====
bj = df_t.loc[df_t['城市'] == '北京'].copy().head(10)
bj['温度_昨天'] = bj['温度(℃)'].shift()
bj['温度变化量'] = bj['温度(℃)'].diff()
bj['温度变化率'] = bj['温度(℃)'].pct_change()

print('北京前10天:')
print(bj[['温度(℃)', '温度_昨天', '温度变化量', '温度变化率']])

bj['温度_明天'] = bj['温度(℃)'].shift(-1)
print('\n昨天 vs 今天 vs 明天:')
print(bj[['温度_昨天', '温度(℃)', '温度_明天']])


## 滑动窗口：rolling()

在时间轴上的滑动窗口计算——移动平均、移动标准差等最常用。

```python
df['MA7'] = df['值'].rolling(window=7).mean()      # 7 日移动平均
df['滚动标准差'] = df['值'].rolling(3).std()        # 3 日波动率
df['滚动最大'] = df['值'].rolling(5).max()          # 5 日最高
```

**对比**：`rolling` = 滑动窗口（重叠），`resample` = 按频率分桶（不重叠）。

In [ ]:
# ===== rolling 滑动窗口 =====
bj = df_t.loc[df_t['城市'] == '北京'].head(20).copy()
print(bj[['温度(℃)']])
bj['MA7'] = bj['温度(℃)'].rolling(window=7, min_periods=1).mean()
bj['MA3'] = bj['温度(℃)'].rolling(window=3, min_periods=1).mean()
#min_periods=1 规定整个窗口只剩一个有效值也要算，在序列开头结尾和 NaN 密集段就不会产出大片 NaN
print('滑动平均（前20天）:')
print(bj[['温度(℃)', 'MA3', 'MA7']])


## 日期范围生成：pd.date_range()

快速生成等间隔日期序列，做时间索引或补全缺失日期。

```python
pd.date_range('2024-01-01', '2024-12-31', freq='D')   # 每天
pd.date_range('2024-01', periods=12, freq='MS')        # 12 个月初
pd.date_range('09:00', '17:00', freq='H')             # 工作小时
```

In [ ]:
# ===== pd.date_range() =====
bdays = pd.date_range('2024-01-01', '2024-12-31', freq='B') #B表示工作日
print(f'2024 年工作日数: {len(bdays)}')
print('前 5 个:', bdays[:5].tolist())

months = pd.date_range('2024-01', periods=12, freq='MS')
print(f'\n12个月月初: {months.tolist()}')
#periods=n，从 start 往后连推 n 个 freq

hours = pd.date_range('2024-06-04 08:00', '2024-06-04 18:00', freq='h')
print(f'\n6月4日每小时: {hours.tolist()}')
